In [1]:
!pip install selenium
!apt-get update
!apt install chromium-chromedriver

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 476.0/476.0 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 2.6 MB/s eta 0:00:00
Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,626 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Get:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Ign:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy Release [5,713 B]
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy Release.gpg [793 B]
Get:10 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [57.5 kB]
Get:11 http://archive.ubuntu.com

In [2]:
import csv
import os
import time
from datetime import date, datetime
from selenium import webdriver
from selenium.webdriver.common.by import By
from concurrent.futures import ThreadPoolExecutor
import shutil
from google.colab import drive

# Montar Google Drive
drive.mount('/content/drive')

# Inicialize o WebDriver
def iniciar_driver():
    opções = webdriver.ChromeOptions()
    opções.add_argument('--headless')
    opções.add_argument('--no-sandbox')
    opções.add_argument('--disable-dev-shm-usage')
    driver = webdriver.Chrome(options=opções)
    driver.implicitly_wait(5)
    return driver

# Função para coletar produtos de uma categoria
def coletar_produtos(categoria, url):
    driver = iniciar_driver()
    produtos = []
    produtos_carregados = set()  # Usaremos um set para rastrear produtos únicos
    pagina_atual = 1

    print(f"Iniciando a coleta de produtos da categoria: {categoria}")

    while True:
        try:
            # Navegue até a página atual
            url_com_pagina = f"{url.split('?')[0]}?page={pagina_atual}"
            driver.get(url_com_pagina)

            # Aguarde até que os produtos sejam carregados
            produtos_atual = driver.find_elements(By.CSS_SELECTOR, '.vtex-search-result-3-x-galleryItem')

            # Adicione os produtos à lista
            novos_produtos = False
            for produto in produtos_atual:
                try:
                    nome_empresa = "Joli"
                    data_coleta = date.today().strftime("%Y-%m-%d")
                    codigo_produto = produto.find_element(By.CSS_SELECTOR, '.vtex-product-summary-2-x-brandName').text.strip()
                    imagem_url = produto.find_element(By.CSS_SELECTOR, '.vtex-product-summary-2-x-image').get_attribute('src')
                    nome_produto = produto.find_element(By.CSS_SELECTOR, '.vtex-product-summary-2-x-productNameContainer').text.strip()
                    moeda = produto.find_element(By.CSS_SELECTOR, '.joli-joli-theme-0-x-price').text.strip()
                    url_produto = produto.find_element(By.CSS_SELECTOR, 'a').get_attribute('href')

                    linha_produto = [nome_empresa, data_coleta, codigo_produto, imagem_url, nome_produto, moeda, url_produto]

                    if nome_produto not in produtos_carregados:
                        produtos.append(linha_produto)
                        produtos_carregados.add(nome_produto)
                        novos_produtos = True

                except:
                    continue

            print(f"Total de produtos coletados de {categoria} até agora: {len(produtos)}")

            if not novos_produtos:
                print(f"Não há mais novos produtos para carregar de {categoria}.")
                break

            pagina_atual += 1

        except Exception:
            print(f"Não foi possível encontrar mais produtos em {categoria}.")
            break

    driver.quit()

    # Salvar os produtos em um arquivo CSV com nome da categoria
    nome_arquivo = os.path.join("output", f'produtos_{categoria}.csv')
    with open(nome_arquivo, 'w', newline='', encoding='utf-8') as arquivo_csv:
        escritor_csv = csv.writer(arquivo_csv)
        escritor_csv.writerow(['Nome empresa', 'Data de coleta', 'Código do produto', 'Imagem', 'Nome do produto', 'Moeda', 'URL do produto'])

        for produto in produtos:
            escritor_csv.writerow(produto)

    print(f"Os produtos da categoria {categoria} foram salvos em: {nome_arquivo}")

# Criar pasta de saída se não existir
output_folder = "output"
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

# Dicionário de URLs por categoria - Notebook 1
urls_notebook_1 = {
    "Cozinhas e areas de servico": "https://www.joli.com.br/banheiros-cozinhas-e-areas-de-servico?page=1",
    "Bomboniere": "https://www.joli.com.br/bomboniere?page=1",
    "Chuveiros e aquecedores": "https://www.joli.com.br/chuveiros-e-aquecedores?page=1",
    "Climatizacao e ventilacao": "https://www.joli.com.br/climatizacao-e-ventilacao?page=1",
    "Decoracao e utilidades domesticas": "https://www.joli.com.br/decoracao-e-utilidades-domesticas?page=1"
}

# Usar múltiplas threads para acelerar a coleta de dados
with ThreadPoolExecutor(max_workers=5) as executor:
    for categoria, url in urls_notebook_1.items():
        executor.submit(coletar_produtos, categoria, url)

# Mover arquivos CSV para o Google Drive
base_destination_dir = '/content/drive/MyDrive/WebScraping/Joli/WebScraping_coleta_Joli'
today = datetime.today().strftime('%Y-%m-%d')
destination_dir = os.path.join(base_destination_dir, today)

if not os.path.exists(destination_dir):
    os.makedirs(destination_dir, exist_ok=True)

source_dir = output_folder

for file_name in os.listdir(source_dir):
    full_file_name = os.path.join(source_dir, file_name)
    if os.path.isfile(full_file_name):
        shutil.copy(full_file_name, destination_dir)

print(f"Arquivos CSV movidos para: {destination_dir}")


Mounted at /content/drive
Iniciando a coleta de produtos da categoria: Bomboniere
Iniciando a coleta de produtos da categoria: Climatizacao e ventilacao
Iniciando a coleta de produtos da categoria: Chuveiros e aquecedores
Iniciando a coleta de produtos da categoria: Cozinhas e areas de servico
Iniciando a coleta de produtos da categoria: Decoracao e utilidades domesticas
Total de produtos coletados de Bomboniere até agora: 6
Total de produtos coletados de Chuveiros e aquecedores até agora: 5
Total de produtos coletados de Decoracao e utilidades domesticas até agora: 5
Total de produtos coletados de Cozinhas e areas de servico até agora: 5
Total de produtos coletados de Climatizacao e ventilacao até agora: 5
Total de produtos coletados de Chuveiros e aquecedores até agora: 8
Total de produtos coletados de Bomboniere até agora: 11
Total de produtos coletados de Cozinhas e areas de servico até agora: 10
Total de produtos coletados de Climatizacao e ventilacao até agora: 10
Total de produt